In [ ]:
!pip install --upgrade --force-reinstall numpy pandas
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

In [ ]:
movies = pd.read_csv('/kaggle/input/datasets/rajmehra03/movielens100k/movies.csv')
ratings = pd.read_csv('/kaggle/input/datasets/rajmehra03/movielens100k/ratings.csv')

print(movies.shape)
print(ratings.shape)

In [ ]:
print("Unique Users:", ratings['userId'].nunique())
print("Unique Movies:", ratings['movieId'].nunique())

In [ ]:
plt.figure()
ratings['rating'].value_counts().sort_index().plot(kind='bar')
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

In [ ]:
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()
num_ratings = len(ratings)

sparsity = 1 - (num_ratings / (num_users * num_movies))
print("Dataset Sparsity:", round(sparsity, 4))

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def content_recommend(title, top_n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]

content_recommend("Toy Story (1995)")

In [ ]:
# Create user-movie matrix
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

# Compute cosine similarity between users
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

def collaborative_recommend(user_id, top_n=5):
    
    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    
    # Weighted average ratings
    weighted_ratings = np.dot(
        similar_users.values,
        user_movie_matrix.loc[similar_users.index]
    )
    
    weighted_ratings = pd.Series(
        weighted_ratings,
        index=user_movie_matrix.columns
    )
    
    # Remove already rated movies
    already_rated = user_movie_matrix.loc[user_id]
    recommendations = weighted_ratings[already_rated == 0]
    
    # Top N
    top_movies = recommendations.sort_values(ascending=False).head(top_n)
    
    return movies[movies['movieId'].isin(top_movies.index)]['title']

collaborative_recommend(1)

In [ ]:
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

In [ ]:
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, 
                                  index=user_movie_matrix.index, 
                                  columns=user_movie_matrix.index)

In [ ]:
def collaborative_recommend(user_id, top_n=5):
    
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    
    weighted_ratings = np.dot(similar_users.values, 
                              user_movie_matrix.loc[similar_users.index])
    
    weighted_ratings = pd.Series(weighted_ratings, 
                                 index=user_movie_matrix.columns)
    
    already_rated = user_movie_matrix.loc[user_id]
    recommendations = weighted_ratings[already_rated == 0]
    
    top_movies = recommendations.sort_values(ascending=False).head(top_n)
    
    return movies[movies['movieId'].isin(top_movies.index)]['title']

collaborative_recommend(1)

In [ ]:
def hybrid_recommend(user_id, title, top_n=5):
    
    content_movies = list(content_recommend(title, top_n=10))
    collab_movies = list(collaborative_recommend(user_id, top_n=10))
    
    # Combine both lists
    combined = content_movies + collab_movies
    
    # Remove duplicates but keep order
    final = list(dict.fromkeys(combined))
    
    return final[:top_n]

print(hybrid_recommend(1, "Toy Story (1995)"))

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.merge(ratings, movies, on='movieId')
data.head()

In [ ]:
user_movie_matrix = data.pivot_table(index='userId',
                                     columns='title',
                                     values='rating')
user_movie_matrix.head()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(user_movie_matrix.fillna(0).T)

similarity_df = pd.DataFrame(movie_similarity,
                             index=user_movie_matrix.columns,
                             columns=user_movie_matrix.columns)

In [ ]:
def recommend_movies(movie_name, num_recommendations=5):
    similar_scores = similarity_df[movie_name].sort_values(ascending=False)
    return similar_scores.iloc[1:num_recommendations+1]

In [ ]:
recommend_movies("Toy Story (1995)")